In [ ]:

# ============================================================
# 1-overtakes.ipynb  ·  F1 超车事件检测与清洗（2018–2026）
#
# 本 Notebook 从 FastF1 正赛数据中识别所有"超车"候选事件，
# 并对每条记录附加 10 维特征，最终输出以下四张表：
#
#   full_overtakes_season   — 所有候选超车（含疑似干扰事件）
#   clean_overtakes_season  — 剔除车队指令 + 越界/锁死后的净超车
#   team_order_season       — 检出的疑似车队指令超车
#   off_track_season        — 检出的越界/锁死/黄旗相关超车
#
# ── 推荐运行顺序 ──────────────────────────────────────────────
#   Cell 1   （本单元）导入依赖库
#   Cell A   配置全局参数与判定阈值
#   Cell D   特征计算辅助函数定义（_calc_* × 10）
#   Cell E   超集列定义 & Schema 强制转换函数
#   Cell G   核心函数 process_one_race() 定义
#   Cell H   多赛季批量处理 & 逐年存盘
#   Cell I   合并各年 CSV → ALL_*.csv
#   Cell J   导出 .dta（Stata）和 .rds（R）格式
# ============================================================
import fastf1
import pandas as pd
import numpy as np
from pprint import pprint

# 禁用响应警告
import warnings
warnings.filterwarnings('ignore')

print("fastf1 版本:", fastf1.__version__)


In [ ]:

# ============================================================
# Cell A — 配置全局参数与判定阈值
#
# ── 车队指令检测阈值 ─────────────────────────────────────────
#   TEAM_ORDER_THROTTLE_THRESHOLD   : Throttle_Drop_Index 上限
#       > 此值 → 超车被追方存在明显收油行为，判定为主动让位
#   TEAM_ORDER_BRAKE_THRESHOLD      : Brake_Pressure_Delta 上限（ms）
#       > 此值 → 两车刹车时序差异过大，判定为主动让刹
#   TEAM_ORDER_CONFIDENCE_THRESHOLD : 置信度下限（0–1）
#       ≥ 此值 → 该超车事件被标记为疑似车队指令
#
# ── 越界/锁死检测阈值 ────────────────────────────────────────
#   OFF_TRACK_VMIN_THRESHOLD        : Vmin_Deviation 上限（%）
#       > 此值 → 被过车手在关键刹车区速度异常偏低，疑似过错
#   OFF_TRACK_CONFIDENCE_THRESHOLD  : 置信度下限（0–1）
#       ≥ 此值 → 该超车事件被标记为越界/锁死相关
#
# ── 其他参数 ────────────────────────────────────────────────
#   TELEMETRY_HZ : telemetry 重采样频率（Hz），降低可加快处理速度
# ============================================================
import os

# ── 全局配置 ─────────────────────────────────────────────────
YEAR = 2024          # 目标赛季（Cell H 使用）

# 判定阈值（可调节）
TEAM_ORDER_THROTTLE_THRESHOLD   = 10.0   # Throttle_Drop_Index > 该值则触发
TEAM_ORDER_BRAKE_THRESHOLD      = 150.0  # Brake_Pressure_Delta(ms) > 该值则触发
TEAM_ORDER_CONFIDENCE_THRESHOLD = 0.5

OFF_TRACK_VMIN_THRESHOLD        = 5.0    # Vmin_Deviation(%) > 该值则触发
OFF_TRACK_CONFIDENCE_THRESHOLD  = 0.5

TELEMETRY_HZ = 10          # 按需 telemetry 重采样频率

# ── 缓存初始化 ────────────────────────────────────────────────
cache_dir = os.path.join(os.getcwd(), "cache")
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

print(f"配置完成：YEAR={YEAR}, 缓存目录={cache_dir}")


In [ ]:

# ============================================================
# Cell D — 特征计算辅助函数定义（共 10 个，供 process_one_race 调用）
#
# ── 依赖的全局变量（由 process_one_race 在调用前写入）──────────
#   tel_store  : Dict[(driver_abbr, lap_n)] → telemetry DataFrame
#   laps_base  : 当场比赛圈速基准表（含 TrackStatus / PitIn / IsAccurate 等）
#   各阈值常量 : 来自 Cell A（TEAM_ORDER_* / OFF_TRACK_*）
#
# ── 函数清单 ──────────────────────────────────────────────────
#   辅助  _get_tel(driver, lap_n)               → 从缓存安全取 telemetry
#   D1    _calc_drs(a, lap_n)                   → Is_DRS_Active      (bool)
#   D2    _calc_rel_dist(a, lap_n)              → RelativeDistance   (float)
#   D3    _calc_throttle_drop(a, b, lap_n)      → Throttle_Drop_Index (float)
#   D4    _calc_brake_delta(a, b, lap_n)        → Brake_Pressure_Delta (ms)
#   D5    _calc_team_order_confidence(tdi, bpd) → TeamOrderConfidence (0–1)
#   D6    _calc_vmin_deviation(b, lap_n)        → Vmin_Deviation     (%)
#   D7    _calc_offtrack_flag(b, lap_n)         → OffTrack_Flag      (bool)
#   D8    _calc_lockup_flag(b, lap_n)           → Lockup_Flag        (bool)
#   D9    _calc_local_yellow(lap_n)             → LocalYellowFlag    (bool)
#   D10   _calc_offtrack_confidence(...)        → OffTrackConfidence (0–1)
# ============================================================

# ── 辅助：从 tel_store 取单圈 telemetry ──────────────────────
def _get_tel(driver: str, lap_n: int):
    t = tel_store.get((driver, lap_n))
    return t if (t is not None and len(t) > 0) else None


# ── D1: Is_DRS_Active ─────────────────────────────────────────
# DRS 值含义：0=不可用,8=可用(DRS关),10=DRS开启
# 取超车发起者（A）在该圈任意时刻 DRS=10 则视为激活
def _calc_drs(driver_a: str, lap_n: int) -> bool:
    t = _get_tel(driver_a, lap_n)
    if t is None or 'DRS' not in t.columns:
        return np.nan
    return bool((t['DRS'] == 10).any())


# ── D2: RelativeDistance ──────────────────────────────────────
# 取超车发起者（A）该圈 RelativeDistance 的中位数作为代表值
def _calc_rel_dist(driver_a: str, lap_n: int) -> float:
    t = _get_tel(driver_a, lap_n)
    if t is None or 'RelativeDistance' not in t.columns:
        return np.nan
    return float(t['RelativeDistance'].median())


# ── D3: Throttle_Drop_Index ──────────────────────────────────
# 被超车手(B)在关键加速段(40-80% RelativeDistance)的收油幅度
# = median(A.Throttle - B.Throttle) 在对齐时间窗内
def _calc_throttle_drop(driver_a: str, driver_b: str, lap_n: int) -> float:
    ta = _get_tel(driver_a, lap_n)
    tb = _get_tel(driver_b, lap_n)
    if ta is None or tb is None:
        return np.nan
    if 'Throttle' not in ta.columns or 'Throttle' not in tb.columns:
        return np.nan
    try:
        ma = ta[['SessionTime', 'Throttle', 'RelativeDistance']].dropna()
        mb = tb[['SessionTime', 'Throttle']].dropna()
        merged = pd.merge_asof(
            ma.sort_values('SessionTime'),
            mb.sort_values('SessionTime'),
            on='SessionTime', direction='nearest',
            suffixes=('_a', '_b')
        )
        seg = merged.query('0.4 <= RelativeDistance <= 0.8') if 'RelativeDistance' in merged.columns else merged
        if len(seg) < 5:
            return np.nan
        return float(np.abs(np.median(seg['Throttle_a'] - seg['Throttle_b'])))
    except Exception:
        return np.nan


# ── D4: Brake_Pressure_Delta ─────────────────────────────────
# 刹车时序差异：检测 Brake=0→1 的上升沿事件，取两车对应事件时间差中位数(ms)
def _calc_brake_delta(driver_a: str, driver_b: str, lap_n: int) -> float:
    ta = _get_tel(driver_a, lap_n)
    tb = _get_tel(driver_b, lap_n)
    if ta is None or tb is None:
        return np.nan
    if 'Brake' not in ta.columns or 'Brake' not in tb.columns:
        return np.nan
    try:
        def brake_events(t):
            b = t['Brake'].astype(int)
            rising = b.diff() > 0
            return list(t.loc[rising, 'SessionTime'])
        ea = brake_events(ta)
        eb = brake_events(tb)
        if len(ea) < 2 or len(eb) < 2:
            return np.nan
        diffs = []
        for t_a in ea:
            closest = min(eb, key=lambda x: abs(pd.Timedelta(x - t_a).total_seconds()))
            dt = abs(pd.Timedelta(t_a - closest).total_seconds()) * 1000
            if dt < 500:
                diffs.append(dt)
        return float(np.median(diffs)) if diffs else np.nan
    except Exception:
        return np.nan


# ── D5: TeamOrderConfidence ───────────────────────────────────
def _calc_team_order_confidence(tdi: float, bpd: float) -> float:
    score = 0.0
    if not np.isnan(tdi):
        score += min(tdi / TEAM_ORDER_THROTTLE_THRESHOLD, 1.0) * 0.6
    if not np.isnan(bpd):
        score += min(bpd / TEAM_ORDER_BRAKE_THRESHOLD, 1.0) * 0.4
    return round(score, 3) if score > 0 else np.nan


# ── D6: Vmin_Deviation ───────────────────────────────────────
def _calc_vmin_deviation(driver_b: str, lap_n: int) -> float:
    tb = _get_tel(driver_b, lap_n)
    if tb is None or 'Speed' not in tb.columns:
        return np.nan
    try:
        yellow_rows = laps_base[
            (laps_base['Driver'] == driver_b) &
            (laps_base['LapNumber'] == lap_n) &
            (laps_base['TrackStatus'].str.contains('2', na=False))
        ]
        if yellow_rows.empty:
            return np.nan
        ref_laps = laps_base[
            (laps_base['Driver'] == driver_b) &
            (laps_base['TrackStatus'] == '1') &
            (laps_base['IsAccurate'] == True)
        ]
        if ref_laps.empty:
            return np.nan
        ref_lap_n = int(ref_laps['LapNumber'].median())
        t_ref = _get_tel(driver_b, ref_lap_n)
        if t_ref is None or 'Speed' not in t_ref.columns:
            return np.nan
        if 'RelativeDistance' not in tb.columns or 'RelativeDistance' not in t_ref.columns:
            return np.nan
        current_min = tb[tb['RelativeDistance'] > 0.7]['Speed'].min()
        ref_min     = t_ref[t_ref['RelativeDistance'] > 0.7]['Speed'].min()
        if pd.isna(ref_min) or ref_min < 1:
            return np.nan
        return round((current_min - ref_min) / ref_min * 100, 2)
    except Exception:
        return np.nan


# ── D7: OffTrack_Flag ────────────────────────────────────────
def _calc_offtrack_flag(driver_b: str, lap_n: int) -> bool:
    tb = _get_tel(driver_b, lap_n)
    if tb is None or 'Status' not in tb.columns:
        return False
    ratio = (tb['Status'] == 'OffTrack').mean()
    return bool(ratio > 0.05)


# ── D8: Lockup_Flag ──────────────────────────────────────────
def _calc_lockup_flag(driver_b: str, lap_n: int) -> bool:
    tb = _get_tel(driver_b, lap_n)
    if tb is None:
        return False
    if 'Brake' not in tb.columns or 'Throttle' not in tb.columns:
        return False
    mask = (tb['Brake'].astype(int) == 1) & (tb['Throttle'] > 30)
    return bool(mask.sum() >= 2)


# ── D9: LocalYellowFlag ──────────────────────────────────────
def _calc_local_yellow(lap_n: int) -> bool:
    rows = laps_base[laps_base['LapNumber'] == lap_n]
    return bool(rows['TrackStatus'].str.contains('2', na=False).any())


# ── D10: OffTrackConfidence ──────────────────────────────────
def _calc_offtrack_confidence(vmin_dev, offtrack, lockup, yellow) -> float:
    score = 0.0
    if not np.isnan(vmin_dev) and vmin_dev > OFF_TRACK_VMIN_THRESHOLD:
        score += 0.4
    if offtrack:
        score += 0.3
    if lockup:
        score += 0.2
    if yellow:
        score += 0.1
    return round(score, 3) if score > 0 else np.nan


print("Cell D: 全部特征计算函数已定义。")


In [ ]:

# ============================================================
# Cell E — 超集列定义 & Schema 强制转换函数
#
#   SUPERSET_COLS        : 单场超车表的 25 个标准字段（不含年份/站次）
#   SEASON_SUPERSET_COLS : 多赛季合并表字段（在前补 Year/Round/EventName）
#
#   enforce_schema(df)   : 确保 df 包含全部超集列
#       · 缺失列自动补 NaN，避免各场之间 merge 时字段错位
#       · 返回按 SUPERSET_COLS 顺序排列的副本
#       · 供 process_one_race（Cell G）调用
# ============================================================

SUPERSET_COLS = [
    'LapNumber', 'Overtaker', 'OvertakerNumber', 'OvertakerTeam',
    'PositionBefore', 'PositionAfter', 'Overtaken', 'OvertakenNumber',
    'OvertakenTeam', 'SessionTime',
    'SameLapWindowStart', 'SameLapWindowEnd',
    'PitInfluenceFlag', 'DNFInfluenceFlag',
    'IsTeammate',
    'Is_DRS_Active', 'RelativeDistance',
    'Throttle_Drop_Index', 'Brake_Pressure_Delta', 'TeamOrderConfidence',
    'Vmin_Deviation', 'OffTrack_Flag', 'Lockup_Flag',
    'LocalYellowFlag', 'OffTrackConfidence',
]

SEASON_SUPERSET_COLS = ['Year', 'Round', 'EventName'] + SUPERSET_COLS

def enforce_schema(df: pd.DataFrame, name: str = '') -> pd.DataFrame:
    """确保 df 包含全部超集列，缺失列以 NaN 填充；返回标准列序的副本。"""
    for col in SUPERSET_COLS:
        if col not in df.columns:
            df = df.assign(**{col: np.nan})
    return df[SUPERSET_COLS].copy()

print("Cell E: SUPERSET_COLS 与 enforce_schema() 已定义。")


In [ ]:

# ============================================================
# Cell G — 核心函数 process_one_race(year, round_n)
#
#   输入  : year（赛季年份）, round_n（站次编号）
#   输出  : full_overtakes DataFrame（含 Year/Round/EventName）
#           失败时打印错误信息并返回 None
#
# ── 集成的处理步骤（原散落于 Cell B/C/F，已整合至本函数）──────
#   Step 1  加载 R session（laps=True, messages=True）
#   Step 2  构建 laps_base + 同圈位置变化检测（原 Cell B）
#           · 逐圈对比 Position 变化，枚举所有 (Overtaker, Overtaken) 对
#           · 过滤 PitOut / DNF 影响的位置变化
#   Step 3  按需加载 telemetry，建立 tel_store 缓存（原 Cell C）
#           · 仅对候选超车涉及的车手 × 圈次加载，节省 API 请求
#   Step 4  批量调用 Cell D 的 _calc_* 函数，附加 10 维特征
#
# ── 全局变量副作用 ────────────────────────────────────────────
#   写入 laps_base（供 D6/D9 读取圈速基准）
#   写入 tel_store（供所有 _calc_* 读取 telemetry）
#
# ── 依赖 ──────────────────────────────────────────────────────
#   Cell A  阈值常量（TEAM_ORDER_* / OFF_TRACK_* / TELEMETRY_HZ）
#   Cell D  全部 _calc_* 辅助函数（须先执行 Cell D）
# ============================================================

def process_one_race(year: int, round_n: int):
    """处理单场正赛，返回 full_overtakes DataFrame（含 Year/Round/EventName）。失败返回 None。"""
    global laps_base, tel_store   # Cell D 的 _calc_* 函数通过全局变量读取这两个对象

    # ── 加载 session ─────────────────────────────────────────
    try:
        sess = fastf1.get_session(year, round_n, 'R')
        sess.load(laps=True, telemetry=False, weather=False, messages=True)
        event_name = sess.event['EventName']
    except Exception as e:
        print(f"  ✗ {year} R{round_n:02d} 加载失败: {e}")
        return None

    # ── B: laps_base + 同圈超车检测 ─────────────────────────
    laps_base = sess.laps[[
        'LapNumber', 'Driver', 'DriverNumber', 'Team',
        'Position', 'PitInTime', 'PitOutTime', 'TrackStatus',
        'LapStartTime', 'Time', 'FastF1Generated', 'IsAccurate',
    ]].copy()
    laps_base['LapNumber'] = laps_base['LapNumber'].astype(int)

    lep = (
        laps_base.dropna(subset=['Position'])
        .assign(Position=lambda d: d['Position'].astype(int))
        .pivot_table(index='LapNumber', columns='Driver', values='Position', aggfunc='first')
        .sort_index()
    )
    lsp = lep.shift(1)

    res = sess.results[['DriverNumber', 'Status', 'Abbreviation']].copy()
    dnf_r = set(res.loc[~res['Status'].str.contains(r'(?i)finished|laps?$', regex=True, na=False), 'Abbreviation'])
    last_p = laps_base.sort_values('LapNumber').groupby('Driver').last()['Position']
    dnf_drv = dnf_r & set(last_p[last_p.isna()].index)

    def _pits(n):
        r = laps_base[laps_base['LapNumber'] == n]
        return set(r.loc[r['PitInTime'].notna(), 'Driver']), set(r.loc[r['PitOutTime'].notna(), 'Driver'])

    dmeta = laps_base[['Driver', 'DriverNumber', 'Team']].drop_duplicates('Driver').set_index('Driver')
    _E = pd.Series({'DriverNumber': '', 'Team': ''})

    records = []
    for lap_n in sorted(lep.index):
        if lap_n == 1 or lap_n not in lsp.index:
            continue
        ps = lsp.loc[lap_n].dropna()
        pe = lep.loc[lap_n].dropna()
        common = ps.index.intersection(pe.index)
        if len(common) < 2:
            continue
        lr = laps_base[laps_base['LapNumber'] == lap_n]
        t_ws, t_we = lr['LapStartTime'].min(), lr['Time'].min()
        pi, po   = _pits(lap_n)
        pip, pop = _pits(lap_n - 1)
        for a in common:
            pa_s, pa_e = ps[a], pe[a]
            if pa_e >= pa_s or a in po or a in pop:
                continue
            ma = dmeta.loc[a] if a in dmeta.index else _E
            for b in common:
                if b == a:
                    continue
                pb_s, pb_e = ps[b], pe[b]
                if not (pa_s > pb_s and pa_e < pb_e):
                    continue
                if b in pi or b in pip or b in dnf_drv:
                    continue
                mb = dmeta.loc[b] if b in dmeta.index else _E
                records.append({
                    'LapNumber': lap_n, 'Overtaker': a,
                    'OvertakerNumber': ma['DriverNumber'], 'OvertakerTeam': ma['Team'],
                    'PositionBefore': int(pa_s), 'PositionAfter': int(pa_e),
                    'Overtaken': b, 'OvertakenNumber': mb['DriverNumber'], 'OvertakenTeam': mb['Team'],
                    'SessionTime': t_we, 'SameLapWindowStart': t_ws, 'SameLapWindowEnd': t_we,
                    'PitInfluenceFlag': False, 'DNFInfluenceFlag': False,
                    'IsTeammate': ma['Team'] == mb['Team'],
                    'Is_DRS_Active': np.nan, 'RelativeDistance': np.nan,
                    'Throttle_Drop_Index': np.nan, 'Brake_Pressure_Delta': np.nan,
                    'TeamOrderConfidence': np.nan, 'Vmin_Deviation': np.nan,
                    'OffTrack_Flag': False, 'Lockup_Flag': False,
                    'LocalYellowFlag': False, 'OffTrackConfidence': np.nan,
                })

    full_ot = pd.DataFrame(records)
    if full_ot.empty:
        print(f"  ○ {year} R{round_n:02d} {event_name}: 无候选超车")
        return full_ot.assign(Year=year, Round=round_n, EventName=event_name)

    # ── C: 按需 telemetry ────────────────────────────────────
    cand_laps = set(full_ot['LapNumber'].unique())
    cand_drvs = set(full_ot['Overtaker']) | set(full_ot['Overtaken'])
    try:
        sess.load(laps=False, telemetry=True, weather=False, messages=False)
    except Exception as e:
        print(f"  ⚠ {year} R{round_n:02d} {event_name} telemetry 失败（跳过特征）: {e}")
        return full_ot.assign(Year=year, Round=round_n, EventName=event_name)

    tel_store = {}
    for drv in cand_drvs:
        drv_laps = sess.laps.pick_drivers(drv)
        for ln in cand_laps:
            rows = drv_laps[drv_laps['LapNumber'] == ln]
            if rows.empty:
                continue
            try:
                car = rows.iloc[0].get_car_data().add_distance().add_relative_distance()
                pos = rows.iloc[0].get_pos_data()
                tel_store[(drv, ln)] = car.merge_channels(pos, frequency=TELEMETRY_HZ)
            except Exception:
                tel_store[(drv, ln)] = None

    # ── D: 特征计算（调用 Cell D 定义的 _calc_* 函数）─────────
    drs_l, rd_l, tdi_l, bpd_l, toc_l = [], [], [], [], []
    vmin_l, ot_l, lk_l, ly_l, otc_l  = [], [], [], [], []
    for _, row in full_ot.iterrows():
        a, b, ln = row['Overtaker'], row['Overtaken'], row['LapNumber']
        tdi = _calc_throttle_drop(a, b, ln)
        bpd = _calc_brake_delta(a, b, ln)
        vmin = _calc_vmin_deviation(b, ln)
        ot   = _calc_offtrack_flag(b, ln)
        lk   = _calc_lockup_flag(b, ln)
        ly   = _calc_local_yellow(ln)
        drs_l.append(_calc_drs(a, ln))
        rd_l.append(_calc_rel_dist(a, ln))
        tdi_l.append(tdi);  bpd_l.append(bpd)
        toc_l.append(_calc_team_order_confidence(tdi, bpd))
        vmin_l.append(vmin); ot_l.append(ot); lk_l.append(lk)
        ly_l.append(ly);     otc_l.append(_calc_offtrack_confidence(vmin, ot, lk, ly))

    full_ot = full_ot.assign(
        Is_DRS_Active=drs_l, RelativeDistance=rd_l,
        Throttle_Drop_Index=tdi_l, Brake_Pressure_Delta=bpd_l, TeamOrderConfidence=toc_l,
        Vmin_Deviation=vmin_l, OffTrack_Flag=ot_l, Lockup_Flag=lk_l,
        LocalYellowFlag=ly_l, OffTrackConfidence=otc_l,
        Year=year, Round=round_n, EventName=event_name,
    )
    print(f"  ✓ {year} R{round_n:02d} {event_name}: {len(full_ot)} 条超车")
    return full_ot

print("process_one_race() 已定义。")


In [ ]:

# ============================================================
# Cell H — 多赛季批量处理 & 逐年存盘
#
# ── 注意事项 ───────────────────────────────────────────────
#   · ALL_YEARS 在此处局部定义，覆盖了 Cell A 的全局值；
#     调整采集范围时修改此处即可，无需回到 Cell A。
#   · SKIP_YEARS 跳过已采集完毕的赛季（当前跳过 2024）。
#   · 每处理完一个赛季立即存盘，防止中途崩溃丢失数据。
#
# ── 输出四张子表（{year} 为实际赛季年份）──────────────────
#   {year}_full_overtakes_season.csv   — 所有候选超车
#   {year}_team_order_season.csv       — 疑似车队指令
#   {year}_off_track_season.csv        — 越界/锁死/黄旗
#   {year}_clean_overtakes_season.csv  — 净超车（剔除后两类）
#
# ── 子表派生逻辑 ────────────────────────────────────────────
#   team_order  ← IsTeammate==True AND TeamOrderConfidence ≥ 阈值
#   off_track   ← (OffTrack/Lockup/LocalYellow 任一为 True)
#                 AND OffTrackConfidence ≥ 阈值
#   clean       ← full 中不属于上述两类的记录
# ============================================================

ALL_YEARS  = list(range(2023, 2026))   # FastF1 数据可用范围（含 2026 进行中赛季）
SKIP_YEARS = {2024}                    # 已爬取，跳过

output_dir = os.path.join(os.getcwd(), "output")
os.makedirs(output_dir, exist_ok=True)

season_summary = []

for year in ALL_YEARS:
    if year in SKIP_YEARS:
        print(f"⊙ 跳过 {year}（已有数据）")
        continue

    # ── 获取赛历 ─────────────────────────────────────────────
    try:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        race_rounds = schedule.loc[schedule['RoundNumber'] > 0, 'RoundNumber'].tolist()
    except Exception as e:
        print(f"⚠ {year} 赛历获取失败: {e}")
        continue

    print(f"\n── {year} 赛季: {len(race_rounds)} 站 ──")

    # ── 逐站爬取 ─────────────────────────────────────────────
    year_frames = []
    for rn in race_rounds:
        df = process_one_race(year, rn)
        if df is not None:
            year_frames.append(df)

    if not year_frames:
        print(f"  {year}: 无有效数据，跳过")
        continue

    # ── 合并当年 full_overtakes ───────────────────────────────
    yr_merged = pd.concat(year_frames, ignore_index=True)
    for col in SEASON_SUPERSET_COLS:
        if col not in yr_merged.columns:
            yr_merged[col] = np.nan
    yr_full = yr_merged[SEASON_SUPERSET_COLS].copy()

    # ── 派生三张子表 ─────────────────────────────────────────
    mask_t = (
        yr_full['IsTeammate'].eq(True) &
        yr_full['TeamOrderConfidence'].fillna(0.0).ge(TEAM_ORDER_CONFIDENCE_THRESHOLD)
    )
    yr_team = yr_full[mask_t].copy()

    mask_o = (
        (yr_full['OffTrack_Flag'].eq(True) |
         yr_full['Lockup_Flag'].eq(True)   |
         yr_full['LocalYellowFlag'].eq(True)) &
        yr_full['OffTrackConfidence'].fillna(0.0).ge(OFF_TRACK_CONFIDENCE_THRESHOLD)
    )
    yr_offtrack = yr_full[mask_o].copy()

    dirty = (
        set(zip(yr_team['Round'],    yr_team['LapNumber'],    yr_team['Overtaker'],    yr_team['Overtaken']))    |
        set(zip(yr_offtrack['Round'], yr_offtrack['LapNumber'], yr_offtrack['Overtaker'], yr_offtrack['Overtaken']))
    )
    mask_c = ~pd.Series(
        list(zip(yr_full['Round'], yr_full['LapNumber'], yr_full['Overtaker'], yr_full['Overtaken'])),
        index=yr_full.index
    ).isin(dirty)
    yr_clean = yr_full[mask_c].copy()

    # ── 立即存盘（防止中途崩溃丢数据）──────────────────────
    for name, df in [
        ('full_overtakes_season',  yr_full),
        ('team_order_season',      yr_team),
        ('off_track_season',       yr_offtrack),
        ('clean_overtakes_season', yr_clean),
    ]:
        path = os.path.join(output_dir, f"{year}_{name}.csv")
        df.to_csv(path, index=False, encoding='utf-8-sig')

    season_summary.append({
        'Year':      year,
        'Rounds':    yr_full['Round'].nunique(),
        'Total':     len(yr_full),
        'TeamOrder': len(yr_team),
        'OffTrack':  len(yr_offtrack),
        'Clean':     len(yr_clean),
    })
    print(f"  ✓ {year}: {len(yr_full)} 条超车 ({yr_full['Round'].nunique()} 站) → 已存盘")

# ── 汇总报告 ─────────────────────────────────────────────────
print("\n\n═══ 多赛季爬取完成 ═══")
if season_summary:
    print(pd.DataFrame(season_summary).to_string(index=False))
else:
    print("无新数据产出。")


In [ ]:

# ============================================================
# Cell I — 合并各年 CSV → ALL_*.csv（全量汇总文件）
#
#   将 output/ 下所有 {year}_{tname}.csv 按通配符合并，
#   包含 Cell H 处理的全部赛季及历史已有数据（如 2024）。
#
#   输出（每张表各一个全量文件）:
#     ALL_full_overtakes_season.csv
#     ALL_clean_overtakes_season.csv
#     ALL_team_order_season.csv
#     ALL_off_track_season.csv
# ============================================================
import glob

output_dir = os.path.join(os.getcwd(), "output")

table_names = [
    'full_overtakes_season',
    'team_order_season',
    'off_track_season',
    'clean_overtakes_season',
]

for tname in table_names:
    pattern = os.path.join(output_dir, f"*_{tname}.csv")
    files = sorted(glob.glob(pattern))
    if not files:
        print(f"  ⚠ 无文件匹配: *_{tname}.csv，跳过")
        continue

    parts = [pd.read_csv(f, encoding='utf-8-sig') for f in files]
    combined = pd.concat(parts, ignore_index=True)

    out_path = os.path.join(output_dir, f"ALL_{tname}.csv")
    combined.to_csv(out_path, index=False, encoding='utf-8-sig')
    years = sorted(combined['Year'].dropna().unique().astype(int)) if 'Year' in combined.columns else []
    print(f"  已合并: ALL_{tname}.csv  ({len(combined)} 行, 赛季: {years})")

print(f"\n输出目录: {output_dir}")


In [ ]:

# ============================================================
# Cell J — 格式转换：导出 Stata .dta 与 R .rds
#
#   将四张 ALL_*.csv 导出为面板数据分析常用格式：
#     · Stata : .dta（version 118，兼容 Stata 14+）
#     · R     : .rds（需 pyreadr 库）
#               若 pyreadr 不可用，降级为 *_for_r.csv（UTF-8 无 BOM）
#
#   目标文件：
#     ALL_clean_overtakes_season.{dta,rds}
#     ALL_full_overtakes_season.{dta,rds}
#     ALL_off_track_season.{dta,rds}
#     ALL_team_order_season.{dta,rds}
# ============================================================

import pyreadr



target_csvs = [
    "ALL_clean_overtakes_season.csv",
    "ALL_full_overtakes_season.csv",
    "ALL_off_track_season.csv",
    "ALL_team_order_season.csv",
]

# 尝试使用 .rds（需要 pyreadr）
try:
    has_pyreadr = True
except Exception:
    has_pyreadr = False
    print("⚠ 未检测到 pyreadr，R 格式将导出为 *_for_r.csv")

for fname in target_csvs:
    csv_path = os.path.join(output_dir, fname)
    if not os.path.exists(csv_path):
        print(f"⚠ 文件不存在，跳过: {csv_path}")
        continue

    df = pd.read_csv(csv_path, encoding="utf-8-sig")

    base = os.path.splitext(fname)[0]

    # 1) 导出 Stata .dta
    dta_path = os.path.join(output_dir, f"{base}.dta")
    df.to_stata(dta_path, write_index=False, version=118)

    # 2) 导出 R 格式
    if has_pyreadr:
        r_path = os.path.join(output_dir, f"{base}.rds")
        pyreadr.write_rds(r_path, df)
        print(f"✓ {fname} -> {os.path.basename(dta_path)} + {os.path.basename(r_path)}")
    else:
        r_csv_path = os.path.join(output_dir, f"{base}_for_r.csv")
        df.to_csv(r_csv_path, index=False, encoding="utf-8")
        print(f"✓ {fname} -> {os.path.basename(dta_path)} + {os.path.basename(r_csv_path)}")


✓ ALL_clean_overtakes_season.csv -> ALL_clean_overtakes_season.dta + ALL_clean_overtakes_season.rds
✓ ALL_full_overtakes_season.csv -> ALL_full_overtakes_season.dta + ALL_full_overtakes_season.rds
✓ ALL_off_track_season.csv -> ALL_off_track_season.dta + ALL_off_track_season.rds
✓ ALL_team_order_season.csv -> ALL_team_order_season.dta + ALL_team_order_season.rds
